In [1]:
import numpy as np
import gymnasium as gym
from typing import Dict, Tuple, Optional
from enum import Enum
from gymnasium.error import DependencyNotInstalled
from envs.stowage_gym import StowageEnv
config_dict = {
    "vessel_shape": (5,5,1),
    "yard_shape": (5,5,1),
    "num_containers": 25,
    "group_num": 3,
    "group_placement": "random",
    "seed": 4307,
    # "env": "SPGE-MC"
}
env = StowageEnv(config=config_dict)

In [ ]:
import torch as th
quantiles = np.array([[[0.1, 0.2, 0.3, 0.4],
                  [0.5, 0.6, 0.7, 0.8]],
                  [[0.9, 1.0, 1.1, 1.2],
                  [1.3, 1.4, 1.5, 1.6]]])
quantiles = th.tensor(quantiles, dtype=th.float32)
action_masks = np.array([[1,1,0,0],[1,1,0,0]])
action_masks = th.tensor(action_masks, dtype=th.bool)
min_value = quantiles.min() - quantiles.max() - 1.0
expanded_masks = action_masks.unsqueeze(1).expand(-1, quantiles.shape[1], -1)
print("expanded_masks", expanded_masks)
quantiles = th.where(expanded_masks, quantiles, min_value * th.ones_like(quantiles))
print("quantiles", quantiles)
print("action_masks", action_masks)
print(quantiles.mean(dim=1))

In [ ]:
from sb3_contrib import QRDQN
import gymnasium as gym
env = gym.make("CartPole-v1")
model = QRDQN("MlpPolicy", env, verbose=1, gamma=0.4)
model.learn(total_timesteps=1000)

In [ ]:
from stable_baselines3 import DQN
import gymnasium as gym
env = gym.make("CartPole-v1")
model = DQN("MlpPolicy", env, verbose=1, gamma=0.4)
model.learn(total_timesteps=1000)

In [2]:
# from algorithms.DQN import MaskableDQN
# from algorithms.A2C import MaskableA2C
from sb3_contrib import QRDQN
from algorithms.DQN import MaskableDQN
from algorithms.QRDQN import MaskableQRDQN
from stable_baselines3 import DQN
# model = MaskableDQN("MlpPolicy", env, verbose=1)
obs, info = env.reset()

model = MaskableQRDQN("MlpPolicy", env, verbose=1, gamma=0.4, )
model.learn(total_timesteps=9000, log_interval=1)

Using cuda device
Wrapping the env with a `Monitor` wrapper
Wrapping the env in a DummyVecEnv.
Episode 1 finished after 26 timesteps
----------------------------------
| rollout/            |          |
|    ep_len_mean      | 25       |
|    ep_rew_mean      | 0        |
|    exploration_rate | 0.45     |
| time/               |          |
|    episodes         | 1        |
|    fps              | 1168     |
|    time_elapsed     | 0        |
|    total_timesteps  | 25       |
----------------------------------
Episode 2 finished after 26 timesteps
----------------------------------
| rollout/            |          |
|    ep_len_mean      | 25       |
|    ep_rew_mean      | 0        |
|    exploration_rate | 0.01     |
| time/               |          |
|    episodes         | 2        |
|    fps              | 1208     |
|    time_elapsed     | 0        |
|    total_timesteps  | 50       |
----------------------------------
Episode 3 finished after 26 timesteps
---------------------

/home/stowage_env/algorithms/policies.py:447: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  action_masks = th.tensor(action_masks, dtype=th.bool).to(quantiles.device)


Episode 5 finished after 26 timesteps
----------------------------------
| rollout/            |          |
|    ep_len_mean      | 25       |
|    ep_rew_mean      | 0        |
|    exploration_rate | 0.01     |
| time/               |          |
|    episodes         | 5        |
|    fps              | 192      |
|    time_elapsed     | 0        |
|    total_timesteps  | 125      |
| train/              |          |
|    learning_rate    | 5e-05    |
|    loss             | 8.54     |
|    n_updates        | 6        |
----------------------------------
Episode 6 finished after 26 timesteps
----------------------------------
| rollout/            |          |
|    ep_len_mean      | 25       |
|    ep_rew_mean      | 0        |
|    exploration_rate | 0.01     |
| time/               |          |
|    episodes         | 6        |
|    fps              | 185      |
|    time_elapsed     | 0        |
|    total_timesteps  | 150      |
| train/              |          |
|    learning_

# SB3

In [ ]:
from envs.stowage_aec import StowageAEC
from wandb.integration.sb3 import WandbCallback
class TestCallback(WandbCallback):
    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)

    def _on_step(self) -> bool:
        # Call the parent class's method to log the metrics
        super()._on_step()
        if self.num_timesteps % 5000 == 0:
            # Evaluate the policy
            rew_lst = []
            total_shifters_lst = []
            current_time_lst = []
            for i in range(5):
                # env = MultiCraneStowageEnv(config=config)
                env = StowageAEC(config_dict)
                obs, info = env.reset()
                rewards = []
                while True:
                    action, _ = self.model.predict(obs, action_masks=env.action_masks())
                    obs, reward, terminated, truncated, info = env.step(action)
                    rewards.append(reward)
                    if terminated or truncated:
                        total_shifters = info.get("total_shifters")
                        current_time = info.get("current_time")
                        break
                mean_r = np.sum(rewards)
                rew_lst.append(mean_r)
                total_shifters_lst.append(total_shifters)
                current_time_lst.append(current_time)
            # Log to tensorboard
            self.logger.record("eval/mean_reward", np.mean(rew_lst))
            self.logger.record("eval/total_shifters", np.mean(total_shifters_lst))
            self.logger.record("eval/current_time", np.mean(current_time_lst))
            self.logger.dump(self.num_timesteps)
        return True

In [ ]:
from algorithms.DQN import MaskableDQN
from algorithms.A2C import MaskableA2C
# model = MaskableDQN("MlpPolicy", env, verbose=1)
env = StowageAEC(config_dict)
obs, info = env.reset()

model = MaskableA2C("MlpPolicy", env, verbose=1, gamma=0.4)
model.learn(total_timesteps=1000)

In [ ]:
from sb3_contrib import MaskablePPO
from envs.stowage_aec import StowageAEC
from envs.stowage_crane_gym import MultiCraneStowageEnv
from algorithms.DQN import MaskableDQN
from algorithms.A2C import MaskableA2C
from algorithms.TRPO import MaskableTRPO
from wandb.integration.sb3 import WandbCallback
from sb3_contrib import TRPO
from stable_baselines3 import PPO, DQN, A2C
import numpy as np
import wandb
for i in range(3):
    for alg in ["DQN", "A2C", "PPO", "TRPO"]:
        env = MultiCraneStowageEnv(config=config_dict)
        # env = StowageAEC(config_dict)
        obs, info = env.reset()
        run = wandb.init(
            project="sb3",
            config=config_dict,
            sync_tensorboard=True,  # auto-upload sb3's tensorboard metrics
            # monitor_gym=True,  # auto-upload the videos of agents playing the game
            # save_code=True,  # optional
        )
        if alg == "DQN":
            model = MaskableDQN("MlpPolicy", env, verbose=0, gamma=0.4,tensorboard_log=f"runs/{run.id}")
        elif alg == "A2C":
            model = MaskableA2C("MlpPolicy", env, verbose=0,gamma=0.4, tensorboard_log=f"runs/{run.id}")
        elif alg == "PPO":
            model = MaskablePPO("MlpPolicy", env, verbose=0,gamma=0.4, tensorboard_log=f"runs/{run.id}")
        elif alg == "TRPO":
            model = MaskableTRPO("MlpPolicy", env, verbose=0,gamma=0.4, tensorboard_log=f"runs/{run.id}")
        model.learn(total_timesteps=10000,callback=WandbCallback())
        run.finish()

In [ ]:
from sb3_contrib import MaskablePPO
from sb3_contrib.common.envs import InvalidActionEnvDiscrete
from sb3_contrib.common.maskable.evaluation import evaluate_policy
from sb3_contrib.common.maskable.utils import get_action_masks
# This is a drop-in replacement for EvalCallback
from sb3_contrib.common.maskable.callbacks import MaskableEvalCallback

env = StowageEnv(config=config_dict)
env.reset()
model = MaskablePPO("MlpPolicy", env, gamma=0.4, seed=32, verbose=1,tensorboard_log="./stow_tensorboard/")
model.learn(100_000,tb_log_name="mask_ppo_300c_8g")

# evaluate_policy(model, env, n_eval_episodes=20, reward_threshold=-10, warn=False)

# model.save("ppo_mask")
# del model # remove to demonstrate saving and loading

# model = MaskablePPO.load("ppo_mask")

# obs, _ = env.reset()

In [ ]:
from algorithms.A2C import MaskableA2C
from sb3_contrib.common.maskable.evaluation import evaluate_policy
from sb3_contrib.common.maskable.utils import get_action_masks
# This is a drop-in replacement for EvalCallback
from sb3_contrib.common.maskable.callbacks import MaskableEvalCallback
env = StowageEnv(config=config_dict)
env.reset()
model = MaskableA2C("MlpPolicy", env, gamma=0.4, seed=32, verbose=1,tensorboard_log="./stow_tensorboard/")
model.learn(100_000,tb_log_name="mask_a2c_300c")

# evaluate_policy(model, env, n_eval_episodes=20, reward_threshold=-10, warn=False)

# model.save("a2c_mask")
# del model # remove to demonstrate saving and loading

# model = MaskableA2C.load("a2c_mask")

# obs, _ = env.reset()

In [ ]:
from algorithms.DQN import MaskableDQN
from sb3_contrib.common.maskable.evaluation import evaluate_policy
env = StowageEnv(config=config_dict)
env.reset()
model = MaskableDQN("MlpPolicy", env, gamma=0.4, verbose=1)
# model.learn(100_000,tb_log_name="maskable_dqn_300c")
model.learn(10)
# model.save("deepq_breakout")

# evaluate_policy(model, env, n_eval_episodes=20, reward_threshold=-10, warn=False)

In [ ]:
from algorithms.TRPO import MaskableTRPO
from sb3_contrib.common.maskable.evaluation import evaluate_policy
env = StowageEnv(config=config_dict)
env.reset()
model = MaskableTRPO("MlpPolicy", env, gamma=0.4, verbose=1)
# model.learn(100_000,tb_log_name="maskable_dqn_300c")
model.learn(10000)

In [ ]:
from stable_baselines3 import DQN

model = DQN("MlpPolicy", env, gamma=0.4, verbose=1, tensorboard_log="./stow_tensorboard/")
env = StowageEnv(config=config_dict)
env.reset()

model.learn(80_000,tb_log_name="dqn")

# evaluate_policy(model, env, n_eval_episodes=20, reward_threshold=-10, warn=False)

# Load SB3 Model

In [ ]:
from algorithms.DQN import MaskableDQN
from sb3_contrib.common.maskable.utils import get_action_masks
model = MaskableDQN.load("deepq_breakout")

obs,_ = env.reset()
v_y_tuples = []
while True:
    action, _states = model.predict(obs, action_masks = get_action_masks(env))
    v_y_tuples.append((env.current_vessel_slot,action))
    observation, reward, terminated, truncated, info = env.step(action)
    if terminated or truncated:
        break
env.close()
print(v_y_tuples)

# Ray

In [ ]:
from ray.rllib.algorithms.ppo import PPOConfig
env = StowageEnv(config_dict)
config = PPOConfig()
config.environment(env = StowageEnv, env_config=config_dict)
config.env_runners(num_env_runners=1)
config.training(
    gamma=0.9, lr=0.01, kl_coeff=0.3, train_batch_size_per_learner=256
)

# Build a Algorithm object from the config and run 1 training iteration.
algo = config.build()
algo.train()